# 01 — Preprocessing

In [ ]:
# Cài đặt thư viện cần thiết trên Google Colab
# !pip -q install pandas openpyxl pyarrow transformers sentencepiece accelerate

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

# CONFIG

DATA_DIR = Path("/content/drive/MyDrive/UIT_Legal_System/Dataset/raw")
OUTPUT_DIR = Path("/content/drive/MyDrive/UIT_Legal_System/Dataset/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train": DATA_DIR / "train.xlsx",
    "val": DATA_DIR / "val.xlsx",
    "test": DATA_DIR / "test.xlsx",
}

EXPECTED_COLUMNS = [
    "index",
    "context",
    "article",
    "document",
    "question",
    "extractive answer",
    "abstractive answer",
    "yes/no",
]

TEXT_COLUMNS = [
    "context",
    "article",
    "document",
    "question",
    "extractive answer",
    "abstractive answer",
]

REQUIRED_NON_EMPTY_COLUMNS = [
    "context",
    "article",
    "document",
    "question",
]

# Chỉ dùng để thống kê token, không dùng để biến đổi dữ liệu đầu ra.
TOKENIZER_MODEL_NAME = "BAAI/bge-m3"

DROP_DUPLICATE_QA_WITHIN_SPLIT = True
CREATE_STRICT_EVAL_SPLITS = True

print("DATA_DIR  :", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_DIR  : /content/drive/MyDrive/UIT_Legal_System/Dataset/raw
OUTPUT_DIR: /content/drive/MyDrive/UIT_Legal_System/Dataset/processed


In [ ]:
def normalize_text(value) -> str:
    """Chuẩn hóa Unicode, khoảng trắng và xuống dòng."""
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("\u00a0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_for_comparison(value) -> str:
    """Chuẩn hóa mạnh hơn để so sánh duplicate/leakage."""
    text = normalize_text(value).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def stable_id(prefix: str, *values: str, length: int = 20) -> str:
    """Tạo ID ổn định từ nội dung, không phụ thuộc thứ tự dòng."""
    payload = "|||".join(normalize_for_comparison(v) for v in values)
    digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:length]
    return f"{prefix}_{digest}"


def json_safe(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def dataframe_to_jsonl(df: pd.DataFrame, path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        for row in df.to_dict(orient="records"):
            record = {key: json_safe(value) for key, value in row.items()}
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_json(data, path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

In [ ]:
raw_splits = {}

for split_name, file_path in FILES.items():
    if not file_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file: {file_path}\n"
            "Hãy đặt train.xlsx, val.xlsx và test.xlsx trong DATA_DIR."
        )

    dataframe = pd.read_excel(file_path)

    missing_columns = [
        column
        for column in EXPECTED_COLUMNS
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Split '{split_name}' thiếu cột: {missing_columns}\n"
            f"Cột hiện có: {list(dataframe.columns)}"
        )

    dataframe = dataframe[EXPECTED_COLUMNS].copy()
    raw_splits[split_name] = dataframe

    print(
        f"{split_name:>5}: "
        f"{len(dataframe):,} rows | "
        f"{len(dataframe.columns)} columns"
    )

print(
    "\nTổng số dòng ban đầu:",
    f"{sum(len(df) for df in raw_splits.values()):,}",
)

train: 7,806 rows | 8 columns
  val: 976 rows | 8 columns
 test: 976 rows | 8 columns

Tổng số dòng ban đầu: 9,758


In [ ]:
# CLEAN TEXT AND CREATE STABLE IDS

processed_splits = {}

for split_name, dataframe in raw_splits.items():
    dataframe = dataframe.copy()

    for column in TEXT_COLUMNS:
        dataframe[column] = dataframe[column].map(normalize_text)

    invalid_mask = pd.Series(False, index=dataframe.index)

    for column in REQUIRED_NON_EMPTY_COLUMNS:
        invalid_mask |= dataframe[column].eq("")

    invalid_count = int(invalid_mask.sum())

    if invalid_count:
        print(
            f"{split_name}: loại {invalid_count:,} dòng "
            "thiếu trường bắt buộc"
        )
        dataframe = dataframe.loc[~invalid_mask].copy()

    dataframe["passage_id"] = [
        stable_id("passage", document, article, context)
        for document, article, context in zip(
            dataframe["document"],
            dataframe["article"],
            dataframe["context"],
        )
    ]

    dataframe["qa_id"] = [
        stable_id("qa", passage_id, question)
        for passage_id, question in zip(
            dataframe["passage_id"],
            dataframe["question"],
        )
    ]

    dataframe["split"] = split_name
    dataframe["source_index"] = dataframe["index"]

    # Model-agnostic: không thêm prefix query:/passage: của E5.
    dataframe["query_text"] = dataframe["question"]
    dataframe["passage_text"] = dataframe["context"]

    before_dedup = len(dataframe)

    if DROP_DUPLICATE_QA_WITHIN_SPLIT:
        dataframe = dataframe.drop_duplicates(
            subset=["qa_id"],
            keep="first",
        ).copy()

    removed_duplicates = before_dedup - len(dataframe)

    dataframe = dataframe.reset_index(drop=True)
    processed_splits[split_name] = dataframe

    print(
        f"{split_name:>5}: {before_dedup:,} -> {len(dataframe):,} "
        f"(duplicate QA removed: {removed_duplicates:,})"
    )

train: 7,806 -> 7,718 (duplicate QA removed: 88)
  val: 976 -> 971 (duplicate QA removed: 5)
 test: 976 -> 976 (duplicate QA removed: 0)


In [ ]:
# DATA QUALITY SUMMARY

summary_rows = []

for split_name, dataframe in processed_splits.items():
    summary_rows.append(
        {
            "split": split_name,
            "rows": len(dataframe),
            "unique_questions": dataframe["question"].nunique(),
            "unique_passages": dataframe["passage_id"].nunique(),
            "unique_documents": dataframe["document"].nunique(),
            "extractive_answers": dataframe[
                "extractive answer"
            ].ne("").sum(),
            "abstractive_answers": dataframe[
                "abstractive answer"
            ].ne("").sum(),
            "yes_no_labeled": dataframe["yes/no"].notna().sum(),
            "avg_question_chars": round(
                dataframe["question"].str.len().mean(),
                2,
            ),
            "avg_context_chars": round(
                dataframe["context"].str.len().mean(),
                2,
            ),
            "max_context_chars": int(
                dataframe["context"].str.len().max()
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

,split,rows,unique_questions,unique_passages,unique_documents,extractive_answers,abstractive_answers,yes_no_labeled,avg_question_chars,avg_context_chars,max_context_chars
0,train,7718,7697,294,21,7718,7718,1261,76.87,1685.42,5482
1,val,971,971,258,21,971,971,141,76.43,1654.62,5482
2,test,976,976,806,20,976,976,156,77.49,1703.86,5482


In [ ]:
# 6. CHECK CROSS-SPLIT LEAKAGE

def overlap_report(left_name: str, right_name: str) -> dict:
    left = processed_splits[left_name]
    right = processed_splits[right_name]

    left_questions = set(
        left["question"].map(normalize_for_comparison)
    )
    right_questions = set(
        right["question"].map(normalize_for_comparison)
    )

    left_qa_ids = set(left["qa_id"])
    right_qa_ids = set(right["qa_id"])

    left_passage_ids = set(left["passage_id"])
    right_passage_ids = set(right["passage_id"])

    return {
        "left_split": left_name,
        "right_split": right_name,
        "exact_question_overlap": len(
            left_questions & right_questions
        ),
        "exact_qa_overlap": len(
            left_qa_ids & right_qa_ids
        ),
        "passage_overlap": len(
            left_passage_ids & right_passage_ids
        ),
    }


leakage_df = pd.DataFrame(
    [
        overlap_report("train", "val"),
        overlap_report("train", "test"),
        overlap_report("val", "test"),
    ]
)

display(leakage_df)

leakage_df.to_csv(
    OUTPUT_DIR / "leakage_report.csv",
    index=False,
    encoding="utf-8-sig",
)

,left_split,right_split,exact_question_overlap,exact_qa_overlap,passage_overlap
0,train,val,38,33,258
1,train,test,15,1,55
2,val,test,5,0,54


- `exact_question_overlap`: cùng một câu hỏi xuất hiện ở hai split.
- `exact_qa_overlap`: cùng câu hỏi và cùng passage xuất hiện ở hai split.
- `passage_overlap`: cùng điều khoản được dùng để tạo nhiều câu hỏi khác nhau.

Với dữ liệu quy định, `passage_overlap` không nhất thiết là lỗi vì một điều
khoản có thể trả lời nhiều câu hỏi. Tuy nhiên, câu hỏi trùng nên được loại
khỏi tập đánh giá strict.

In [ ]:
# CREATE STRICT VALIDATION AND TEST SPLITS

if CREATE_STRICT_EVAL_SPLITS:
    train_question_keys = set(
        processed_splits["train"]["question"].map(
            normalize_for_comparison
        )
    )

    strict_val = processed_splits["val"].loc[
        ~processed_splits["val"]["question"]
        .map(normalize_for_comparison)
        .isin(train_question_keys)
    ].copy()

    train_and_val_question_keys = (
        train_question_keys
        | set(
            processed_splits["val"]["question"].map(
                normalize_for_comparison
            )
        )
    )

    strict_test = processed_splits["test"].loc[
        ~processed_splits["test"]["question"]
        .map(normalize_for_comparison)
        .isin(train_and_val_question_keys)
    ].copy()

    processed_splits["strict_val"] = strict_val.reset_index(drop=True)
    processed_splits["strict_test"] = strict_test.reset_index(drop=True)

    print("val          :", len(processed_splits["val"]))
    print("strict_val   :", len(processed_splits["strict_val"]))
    print("test         :", len(processed_splits["test"]))
    print("strict_test  :", len(processed_splits["strict_test"]))

val          : 971
strict_val   : 933
test         : 976
strict_test  : 958


In [ ]:
# BUILD A UNIQUE PASSAGE CORPUS

original_split_names = ["train", "val", "test"]

all_qa = pd.concat(
    [processed_splits[name] for name in original_split_names],
    ignore_index=True,
)

corpus = (
    all_qa[
        [
            "passage_id",
            "document",
            "article",
            "context",
        ]
    ]
    .drop_duplicates(subset=["passage_id"], keep="first")
    .reset_index(drop=True)
)

# Dùng context làm nội dung gốc.
corpus["content"] = corpus["context"]

# Dùng metadata + context cho embedding để phân biệt các điều khoản gần nhau.
corpus["embedding_text"] = (
    "Văn bản: "
    + corpus["document"]
    + "\nĐiều khoản: "
    + corpus["article"]
    + "\nNội dung: "
    + corpus["context"]
)

corpus["title"] = (
    corpus["document"]
    + " - "
    + corpus["article"]
)

qa_count = (
    all_qa.groupby("passage_id")
    .size()
    .rename("qa_count")
)

corpus = corpus.merge(
    qa_count,
    on="passage_id",
    how="left",
)

corpus["content_hash"] = corpus["context"].map(
    lambda text: stable_id("content", text)
)

print("Số passage duy nhất:", f"{len(corpus):,}")
display(corpus.head(3))

Số passage duy nhất: 1,045


,passage_id,document,article,context,content,embedding_text,title,qa_count,content_hash
0,passage_7a4034bfcca9a2af6225,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂN...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG - Điều ...,65,content_7fce8ab9b5ca93015231
1,passage_a9f2bb6e5bcaf05aa8c3,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Văn bản: QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ...,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,61,content_8ce05a028a6c8e694978
2,passage_4a0ee8a7dcf2c8b4a016,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO,Điều 5. Chương trình đào tạo,Điều 5. Chương trình đào tạo\nCT CLC được xây ...,Điều 5. Chương trình đào tạo\nCT CLC được xây ...,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯ...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO -...,37,content_2b671f3959191064a689


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_MODEL_NAME,
    trust_remote_code=True,
)

def token_length(text: str) -> int:
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False,
        )
    )

token_stats_source = all_qa[
    ["question", "context"]
].drop_duplicates().copy()

token_stats_source["question_tokens"] = (
    token_stats_source["question"].map(token_length)
)

token_stats_source["context_tokens"] = (
    token_stats_source["context"].map(token_length)
)

corpus["embedding_text_tokens"] = (
    corpus["embedding_text"].map(token_length)
)

token_summary = pd.DataFrame(
    {
        "question_tokens": token_stats_source[
            "question_tokens"
        ].describe(
            percentiles=[0.5, 0.9, 0.95, 0.99]
        ),
        "context_tokens": token_stats_source[
            "context_tokens"
        ].describe(
            percentiles=[0.5, 0.9, 0.95, 0.99]
        ),
        "embedding_text_tokens": corpus[
            "embedding_text_tokens"
        ].describe(
            percentiles=[0.5, 0.9, 0.95, 0.99]
        ),
    }
)

display(token_summary)

token_summary.to_csv(
    OUTPUT_DIR / "token_length_summary.csv",
    encoding="utf-8-sig",
)

token_stats_source.to_parquet(
    OUTPUT_DIR / "token_length_details.parquet",
    index=False,
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

,question_tokens,context_tokens,embedding_text_tokens
count,9616.000000,9616.000000,1045.000000
mean,22.340890,487.184276,470.227751
std,8.439442,355.824754,324.932168
min,5.000000,27.000000,75.000000
50%,21.000000,372.000000,355.000000
90%,33.000000,1086.000000,954.000000
95%,39.000000,1223.000000,1195.000000
99%,49.000000,1447.000000,1501.560000
max,86.000000,1675.000000,1761.000000


## Chọn độ dài tối đa

Không nên chọn `max_length` chỉ dựa trên khả năng tối đa của model.

Sau khi chạy cell thống kê token:

- chọn `max_query_length` gần percentile 99 của câu hỏi;
- chọn `max_passage_length` gần percentile 95 hoặc 99 của passage;
- kiểm tra tỷ lệ bị truncate trước khi huấn luyện.

Cấu hình khởi đầu thường hợp lý:

```python
max_query_length = 128
max_passage_length = 512
```

Nếu nhiều passage vượt 512 token, notebook huấn luyện có thể tăng lên
768 hoặc 1024 dựa trên VRAM và tốc độ mong muốn.

In [ ]:
# EXPORT MODEL-AGNOSTIC RETRIEVER DATA


retriever_columns = [
    "qa_id",
    "passage_id",
    "question",
    "context",
    "query_text",
    "passage_text",
    "document",
    "article",
    "extractive answer",
    "abstractive answer",
    "yes/no",
    "source_index",
    "split",
]

retriever_splits = {}

for split_name, dataframe in processed_splits.items():
    export_df = dataframe[retriever_columns].copy()
    retriever_splits[split_name] = export_df

    dataframe_to_jsonl(
        export_df,
        OUTPUT_DIR / f"{split_name}_retriever.jsonl",
    )

    export_df.to_parquet(
        OUTPUT_DIR / f"{split_name}_retriever.parquet",
        index=False,
    )

print("Đã xuất dữ liệu retriever cho tất cả split.")

Đã xuất dữ liệu retriever cho tất cả split.


In [ ]:
# EXPORT QUERIES AND QRELS FOR IR EVALUATION

evaluation_splits = [
    name
    for name in [
        "val",
        "test",
        "strict_val",
        "strict_test",
    ]
    if name in retriever_splits
]

for split_name in evaluation_splits:
    dataframe = retriever_splits[split_name]

    queries = {
        row.qa_id: row.question
        for row in dataframe.itertuples(index=False)
    }

    qrels = {}

    for row in dataframe.itertuples(index=False):
        qrels.setdefault(row.qa_id, [])
        if row.passage_id not in qrels[row.qa_id]:
            qrels[row.qa_id].append(row.passage_id)

    save_json(
        queries,
        OUTPUT_DIR / f"{split_name}_queries.json",
    )

    save_json(
        qrels,
        OUTPUT_DIR / f"{split_name}_qrels.json",
    )

print("Đã xuất queries và qrels.")

Đã xuất queries và qrels.


In [ ]:
# EXPORT GENERATOR INSTRUCTION DATA

SYSTEM_PROMPT = (
    "Bạn là trợ lý tư vấn quy định trường học. "
    "Chỉ trả lời dựa trên nội dung quy định được cung cấp. "
    "Không tự bổ sung điều kiện, thời hạn, mức xử lý hoặc ngoại lệ. "
    "Nếu tài liệu không đủ căn cứ, hãy nói rõ chưa đủ thông tin. "
    "Khi có thể, hãy nêu tên văn bản và điều khoản liên quan."
)

def choose_target_answer(row: dict) -> str:
    abstractive_answer = normalize_text(
        row.get("abstractive answer", "")
    )
    extractive_answer = normalize_text(
        row.get("extractive answer", "")
    )

    return abstractive_answer or extractive_answer

def build_generator_record(row: dict) -> dict:
    user_message = (
        f"Văn bản: {row['document']}\n"
        f"Điều khoản: {row['article']}\n\n"
        f"Nội dung quy định:\n{row['context']}\n\n"
        f"Câu hỏi: {row['question']}"
    )

    return {
        "qa_id": row["qa_id"],
        "passage_id": row["passage_id"],
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
            {
                "role": "assistant",
                "content": choose_target_answer(row),
            },
        ],
        "metadata": {
            "document": row["document"],
            "article": row["article"],
            "source_index": json_safe(row["source_index"]),
            "split": row["split"],
        },
    }

for split_name in ["train", "val", "test"]:
    output_path = OUTPUT_DIR / f"{split_name}_generator.jsonl"

    with output_path.open("w", encoding="utf-8") as file:
        for row in processed_splits[split_name].to_dict(
            orient="records"
        ):
            record = build_generator_record(row)
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

print("Đã xuất dữ liệu generator.")

Đã xuất dữ liệu generator.


In [ ]:
# EXPORT CORPUS AND MASTER TABLE

corpus_export_columns = [
    "passage_id",
    "title",
    "document",
    "article",
    "content",
    "embedding_text",
    "embedding_text_tokens",
    "qa_count",
    "content_hash",
]

corpus_export = corpus[corpus_export_columns].copy()

dataframe_to_jsonl(
    corpus_export,
    OUTPUT_DIR / "corpus.jsonl",
)

corpus_export.to_parquet(
    OUTPUT_DIR / "corpus.parquet",
    index=False,
)

master_columns = [
    "qa_id",
    "passage_id",
    "split",
    "source_index",
    "document",
    "article",
    "question",
    "context",
    "extractive answer",
    "abstractive answer",
    "yes/no",
]

all_qa[master_columns].to_parquet(
    OUTPUT_DIR / "qa_master.parquet",
    index=False,
)

print("Đã xuất corpus và QA master.")

Đã xuất corpus và QA master.


In [ ]:
# SAVE PREPROCESSING CONFIG

preprocessing_config = {
    "schema_version": "2.0",
    "model_agnostic": True,
    "source_files": {
        name: str(path)
        for name, path in FILES.items()
    },
    "tokenizer_used_for_statistics": TOKENIZER_MODEL_NAME,
    "retriever_input_policy": {
        "query_text": "raw normalized question",
        "passage_text": "raw normalized context",
        "embedding_text": (
            "document + article + context; "
            "benchmark against context-only later"
        ),
        "model_specific_instruction": (
            "must be added in training/inference code"
        ),
    },
    "duplicate_policy": {
        "within_split_qa_duplicates_removed": (
            DROP_DUPLICATE_QA_WITHIN_SPLIT
        ),
        "cross_split_passage_duplicates_removed": False,
        "strict_eval_splits_created": (
            CREATE_STRICT_EVAL_SPLITS
        ),
    },
}

save_json(
    preprocessing_config,
    OUTPUT_DIR / "preprocessing_config.json",
)

print(
    json.dumps(
        preprocessing_config,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "schema_version": "2.0",
  "model_agnostic": true,
  "source_files": {
    "train": "/content/drive/MyDrive/UIT_Legal_System/Dataset/raw/train.xlsx",
    "val": "/content/drive/MyDrive/UIT_Legal_System/Dataset/raw/val.xlsx",
    "test": "/content/drive/MyDrive/UIT_Legal_System/Dataset/raw/test.xlsx"
  },
  "tokenizer_used_for_statistics": "Qwen/Qwen3-Embedding-4B",
  "retriever_input_policy": {
    "query_text": "raw normalized question",
    "passage_text": "raw normalized context",
    "embedding_text": "document + article + context; benchmark against context-only later",
    "model_specific_instruction": "must be added in training/inference code"
  },
  "duplicate_policy": {
    "within_split_qa_duplicates_removed": true,
    "cross_split_passage_duplicates_removed": false,
    "strict_eval_splits_created": true
  }
}


In [ ]:
# FINAL VALIDATION

required_outputs = [
    "corpus.jsonl",
    "corpus.parquet",
    "qa_master.parquet",
    "train_retriever.jsonl",
    "train_retriever.parquet",
    "val_retriever.jsonl",
    "val_retriever.parquet",
    "test_retriever.jsonl",
    "test_retriever.parquet",
    "train_generator.jsonl",
    "val_generator.jsonl",
    "test_generator.jsonl",
    "val_queries.json",
    "val_qrels.json",
    "test_queries.json",
    "test_qrels.json",
    "dataset_summary.csv",
    "leakage_report.csv",
    "token_length_summary.csv",
    "token_length_details.parquet",
    "preprocessing_config.json",
]

if CREATE_STRICT_EVAL_SPLITS:
    required_outputs.extend(
        [
            "strict_val_retriever.jsonl",
            "strict_val_retriever.parquet",
            "strict_test_retriever.jsonl",
            "strict_test_retriever.parquet",
            "strict_val_queries.json",
            "strict_val_qrels.json",
            "strict_test_queries.json",
            "strict_test_qrels.json",
        ]
    )

missing_outputs = [
    filename
    for filename in required_outputs
    if not (OUTPUT_DIR / filename).exists()
]

if missing_outputs:
    raise RuntimeError(
        "Thiếu file đầu ra:\n- "
        + "\n- ".join(missing_outputs)
    )

print("Tất cả file bắt buộc đã được tạo.\n")

for path in sorted(OUTPUT_DIR.glob("*")):
    size_kb = path.stat().st_size / 1024
    print(f"{path.name:40s} {size_kb:12.2f} KB")

Tất cả file bắt buộc đã được tạo.

corpus.jsonl                                  4634.75 KB
corpus.parquet                                1046.29 KB
dataset_summary.csv                              0.32 KB
leakage_report.csv                               0.13 KB
preprocessing_config.json                        0.82 KB
qa_master.parquet                             2342.40 KB
strict_test_qrels.json                          66.43 KB
strict_test_queries.json                       128.33 KB
strict_test_retriever.jsonl                   5184.59 KB
strict_test_retriever.parquet                  625.33 KB
strict_val_qrels.json                           64.69 KB
strict_val_queries.json                        124.42 KB
strict_val_retriever.jsonl                    4930.90 KB
strict_val_retriever.parquet                   621.16 KB
test_generator.jsonl                          3470.24 KB
test_qrels.json                                 67.67 KB
test_queries.json                              129.80